# 2026-06-05 Korean Tokenization Practice

오늘 실습 목표는 같은 한국어 문장을 여러 방식으로 토큰화하고 결과 차이를 비교하는 것이다.

비교할 방식:

- 공백 기반 토큰화
- 정규식 기반 토큰화
- Okt 형태소 분석
- Kkma 형태소 분석
- 형태소 기반 TF-IDF + 코사인 유사도

In [ ]:
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)

## 1. 예제 문장 준비

한국어 토큰화에서 자주 문제가 되는 조사, 어미, 복합명사, 신조어, 숫자/단위, 반복 문자를 포함한다.

In [ ]:
sentences = [
    "학교에서 자연어처리를 공부하였습니다.",
    "고시원 단기 입실 가능한 방 있나요?",
    "미용실 예약 페이지가 필요해요.",
    "ㅋㅋㅋㅋ 이 서비스 진짜 편하네요!",
    "3개월 계약 가능한가요? 보증금은 30만원입니다.",
    "자연어 처리와 자연어처리는 검색 결과가 다르게 나올 수 있습니다.",
]

pd.DataFrame({"sentence": sentences})

## 2. 공백 기반 토큰화

가장 단순한 방식이다. 빠르지만 한국어 조사/어미/복합명사를 제대로 분리하지 못한다.

In [ ]:
def whitespace_tokenize(text: str):
    return text.split()

space_rows = [{"sentence": s, "tokens": whitespace_tokenize(s), "token_count": len(whitespace_tokenize(s))} for s in sentences]
pd.DataFrame(space_rows)

## 3. 정규식 기반 토큰화

한글, 영어, 숫자 단위를 중심으로 토큰을 추출한다. 특수문자 처리 규칙을 직접 설계할 수 있다.

In [ ]:
TOKEN_PATTERN = re.compile(r"[가-힣]+|[A-Za-z]+|\d+(?:\.\d+)?|[ㅋㅎㅠㅜ]+")

def regex_tokenize(text: str):
    return TOKEN_PATTERN.findall(text)

regex_rows = [{"sentence": s, "tokens": regex_tokenize(s), "token_count": len(regex_tokenize(s))} for s in sentences]
pd.DataFrame(regex_rows)

## 4. KoNLPy 설치/환경 확인

Okt/Kkma는 `konlpy`와 Java 환경이 필요하다. 설치되어 있지 않으면 아래 셀은 안내 메시지만 보여준다.

In [ ]:
try:
    from konlpy.tag import Okt, Kkma
    KONLPY_AVAILABLE = True
    print("KoNLPy 사용 가능")
except Exception as e:
    KONLPY_AVAILABLE = False
    print("KoNLPy 사용 불가:", e)
    print("필요 시 터미널에서 설치: pip install konlpy")

## 5. Okt 형태소 분석

Okt는 설치와 사용이 비교적 쉽고 SNS/구어체 문장에 자주 사용된다.

In [ ]:
if KONLPY_AVAILABLE:
    okt = Okt()
    okt_rows = []
    for s in sentences:
        okt_rows.append({
            "sentence": s,
            "morphs": okt.morphs(s),
            "pos": okt.pos(s),
            "nouns": okt.nouns(s),
        })
    display(pd.DataFrame(okt_rows))
else:
    print("KoNLPy 환경이 준비되면 다시 실행하세요.")

## 6. Kkma 형태소 분석

Kkma는 품사 태그가 세밀하고 복합명사 분석에 강하지만 속도가 느릴 수 있다.

In [ ]:
if KONLPY_AVAILABLE:
    kkma = Kkma()
    kkma_rows = []
    for s in sentences[:3]:
        kkma_rows.append({
            "sentence": s,
            "morphs": kkma.morphs(s),
            "pos": kkma.pos(s),
            "nouns": kkma.nouns(s),
        })
    display(pd.DataFrame(kkma_rows))
else:
    print("KoNLPy 환경이 준비되면 다시 실행하세요.")

## 7. 토큰화 결과 비교

각 방식의 토큰 수와 토큰 결과를 비교한다.

In [ ]:
compare_rows = []
for s in sentences:
    row = {
        "sentence": s,
        "space_tokens": whitespace_tokenize(s),
        "regex_tokens": regex_tokenize(s),
        "space_count": len(whitespace_tokenize(s)),
        "regex_count": len(regex_tokenize(s)),
    }
    if KONLPY_AVAILABLE:
        row["okt_morphs"] = okt.morphs(s)
        row["okt_count"] = len(okt.morphs(s))
    compare_rows.append(row)

pd.DataFrame(compare_rows)

## 8. 형태소 기반 TF-IDF + 코사인 유사도

고시원 FAQ 추천을 가정하고, 문의 문장과 FAQ 후보 문장의 유사도를 계산한다.

In [ ]:
faq_texts = [
    "단기 입실 가능한 방이 있나요",
    "단기 계약도 가능한가요",
    "보증금은 얼마인가요",
    "월세 납부일을 알고 싶어요",
    "방 청소나 수리 요청은 어디에 하나요",
]

query = "고시원 단기 입실 가능한 방 있나요?"

def tokenizer_for_tfidf(text: str):
    if KONLPY_AVAILABLE:
        return okt.morphs(text, stem=True)
    return regex_tokenize(text)

vectorizer = TfidfVectorizer(tokenizer=tokenizer_for_tfidf, token_pattern=None)
matrix = vectorizer.fit_transform([query] + faq_texts)
scores = cosine_similarity(matrix[0:1], matrix[1:]).flatten()

result = pd.DataFrame({
    "query": query,
    "faq": faq_texts,
    "similarity": scores,
}).sort_values("similarity", ascending=False)

result

## 9. 오늘 정리

- 한국어는 공백 기반 토큰화만으로 충분하지 않다.
- 형태소 분석은 의미 단위와 품사 정보를 보존한다.
- 서브워드는 OOV 문제를 줄이고 프리트레인 모델과 호환된다.
- 토큰화 방식은 검색, 추천, 분류, FAQ 자동화 품질에 직접 영향을 준다.
- Roomy Project에서는 입주 문의/FAQ 추천 실험으로 연결할 수 있다.